# 实验 2：解剖 Qwen3-0.6B 的一次完整前向传播

## 今天回答的问题

**Qwen3-0.6B 对一段真实 token 序列究竟进行了哪些计算，这些计算能否被我们自己重新实现并验证？**

实验 1 画出了模型的外部骨架，也量了五个顶层阶段的输出形状，但每个 Block 内部仍是黑盒。
本实验把黑盒打开：从一个真实句子出发，沿真实 forward 路径走完 28 个 Decoder Layer，
每一步都自己实现一遍并与官方对齐，直到 logits 和 Top-K 预测。

本实验采用**两遍结构**：

- **第一遍：搭脚手架。** 运行官方 `Qwen3ForCausalLM` 的 forward 路径，
  把每个关键节点的中间状态以及模型的内部参数记录下来。这一遍不解释任何计算。
- **第二遍：复现。** 用 PyTorch 基础算子重写每一个模块，参数全部取自真实模型，
  然后与第一遍的记录逐节点对齐。

### 本实验不做的事

- 不研究 tokenizer 如何切词。它只作为入口工具：`自然语言 → tokenizer → input_ids`，从 `input_ids` 开始解剖。
- 不讨论"Qwen3 为什么这样设计"，只弄清"它究竟怎么算"。
- 不使用 KV Cache，只研究完整序列的一次 forward（prefill）。
- 不涉及训练、梯度、采样随机性。

### 真实性原则

不凭记忆猜测计算过程。任何一句关于"它怎么算"的话，都要能指回一个可以当场查的来源。
四类问题各有各的来源：

| 想知道的事 | 去问 | 本文的例子 |
|---|---|---|
| 某个超参是多少 | 本地 `config.json` | `head_dim=128` 是写死的，不是 `hidden_size / n_heads` |
| 磁盘上究竟存了哪些张量 | 本地权重文件 | `lm_head.weight` 根本没存，因为它与 embedding 共享 |
| 某个算子的计算顺序 | 实际 Transformers 源码 | `q_norm` 作用在 `head_dim=128` 上，不是 1024 |
| 某个节点的数值到底是什么 | 官方模型跑一遍的结果 | 每个 `check` 的比对基准 |

四者互不替代：源码说不出参数量，配置说不出计算顺序。能互相印证的地方就印证 ——
`q_norm` 归一化的单位是 128 维，源码里能读出来，权重文件里那个 `[128]` 的形状也能佐证。

每个重要模块都会记录类名、函数名和源码行号，形成证据链（见 §0.5）。

## 0. 环境与实验对象核对

正式开始前把"解剖对象"钉死：哪一个模型文件、哪一份源码、哪一种 dtype、哪些维度常量。
后面所有"源码位置"都指向 0.6 里那份 `modeling_qwen3.py`，
所有形状都从 0.5 的常量取，不在代码里写死数字。

### 0.1 导入与工具箱

实现细节都在 `notebooks/qwen_kit.py`，正文只导入八个名字：`show` 出表格，
`diff_config` 比两份配置，`observe` / `params` 建两份目录，`look` / `check` 是两个动作，
`record` / `summary` 管汇总。
这些都是脚手架——表格怎么排、hook 怎么挂、误差怎么算，
知道怎么用就够了，读实现不会加深对模型的理解。

模型路径和源码位置由工具箱在导入时定位（往上找到含 `models/Qwen3-0.6B-Base` 的目录），
`show_path` 负责把它们打成项目相对路径，输出里不带绝对路径。


In [114]:
import json
from math import prod

import torch
import transformers
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3 import modeling_qwen3 as Q3

# 工具箱在 notebooks/qwen_kit.py。搬出去是因为这些都是脚手架，不是实验内容：
# 表格怎么排、hook 怎么挂、误差怎么算，知道怎么用就够，读它不会加深对模型的理解。
from qwen_kit import (MODEL_PATH, REL_THRESHOLD, SOURCE_PATH,
                      check, diff_config, look, observe, params, record,
                      setup, show, show_evidence, show_path, summary)


In [115]:
show([('torch', torch.__version__),
      ('transformers', transformers.__version__),
      ('模型路径', show_path(MODEL_PATH)),
      ('源码位置', show_path(SOURCE_PATH))],
     header=['环境', '值'])


环境,值
torch,2.13.0+cu130
transformers,5.15.1
模型路径,models/Qwen3-0.6B-Base
源码位置,.venv/lib/python3.14/site-packages/transformers/models/qwen3/modeling_qwen3.py


### 0.2 两个加载决定：float32 与 eager

**为什么 float32**：磁盘上的权重是 `bfloat16`，但 bf16 精度不够，看不出复现是对是错。

本实验判定"复现一致"的标准是相对误差 < `1e-5`。而 bf16 只有 7 位尾数，
一次 1024 维 matmul 的舍入误差实测就有 `3.75e-03` —— 比标准大两个半数量级。
在这个精度下，两份完全等价的实现也能差出 1e-3，误差是"写错了"还是"bf16 就这样"
根本分不出来。float32 有 23 位尾数，同样的运算误差 `3.52e-07`，比标准低两个数量级：
超出 1e-5 就一定是实现有问题。

而且源码本来就这么干：RMSNorm 算方差、attention 的 softmax、RoPE 算 cos/sin，
都在内部强制转成 float32。关键计算从来不用 bf16。

**为什么 eager**：默认的 sdpa 路径把整个 attention 打包进一个融合算子，中间量看不到。
本实验要逐个节点比对，所以要 eager 这条用基础算子拼出来的路径。实测两者的差别：

| | sdpa | eager |
|---|---|---|
| `out.attentions` | 空的 | 28 个 `[1, 16, S, S]` 张量 |
| attention 收到的 mask | `None` | `[1, 1, S, S]` 张量 |

sdpa 不给 attention weights，causal mask 也不构造（用 `is_causal` 标志让底层自己处理）。
这两样正是 §7.6 要看的东西。


In [116]:
DTYPE = torch.float32
DEVICE = 'cpu'

# 全程关闭梯度。本实验只研究推理，不涉及训练。
# 注意这里用 set_grad_enabled 而不是 torch.inference_mode()：
# inference_mode 产生的张量不能再参与后续（被 autograd 追踪的）计算，
# 而我们要用第一遍抓到的张量喂给第二遍的复现代码。
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=DTYPE,
    attn_implementation='eager',
).to(DEVICE).eval()

config = model.config

# 工具箱的维度常量和字段表都从 model.config 现推，所以要等模型加载完才能建。
setup(model, tokenizer)

show([('模型类', type(model).__name__, ''),
      ('device', str(next(model.parameters()).device), ''),
      ('dtype', str(next(model.parameters()).dtype), '见上'),
      ('attn 实现', config._attn_implementation, '见上'),
      ('已切到 eval（training=False）', not model.training, '布尔列 ✓ 表示确认通过'),
      ('已关闭梯度', not torch.is_grad_enabled(), '见上面的 set_grad_enabled'),
      ('参数量', f'{sum(p.numel() for p in model.parameters()):,}', '')],
     header=['加载结果', '值', '说明'])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

加载结果,值,说明
模型类,Qwen3ForCausalLM,
device,cpu,
dtype,torch.float32,见上
attn 实现,eager,见上
已切到 eval（training=False）,✓,布尔列 ✓ 表示确认通过
已关闭梯度,✓,见上面的 set_grad_enabled
参数量,"596,049,920",


### 0.3 两份配置：磁盘上的 config.json 与 config 对象

`config.json` 是训练时定下的超参，`model.config` 是 transformers 解析后的对象。
两者不是一份东西的两个副本：键会改名、会被收进子字典，值也可能被加载参数顶掉。
后面每个维度常量都从 config 对象读，所以差异要先找清楚。

两边完整列出来，读法照 git diff：

| 标记 | 含义 |
|---|---|
| (空) | 两边一致，整行调暗 |
| `-` | `config.json` 这边的值 |
| `+` | `config` 对象这边的值 |


In [117]:
raw_config = json.loads((MODEL_PATH / 'config.json').read_text())

# 改名 / 搬家这类差异，光看值对不上还不知道发生了什么，逐条注明。
RENAMED = {'torch_dtype':     '已废弃，改名 dtype',
           'rope_scaling':    '改名 rope_parameters',
           'rope_theta':      '收进 rope_parameters 字典',
           'dtype':           'torch_dtype 的新名字，值由加载参数决定',
           'rope_parameters': 'rope_theta / rope_scaling 的落脚处',
           'layer_types':     '每层用哪种 attention，由 config 推出来的'}

diff_config(raw_config, config, notes=RENAMED);

,key,value,说明
,architectures,['Qwen3ForCausalLM'],
,attention_bias,False,
,attention_dropout,0.0,
,bos_token_id,151643,
,eos_token_id,151643,
,head_dim,128,
,hidden_act,'silu',
,hidden_size,1024,
,initializer_range,0.02,
,intermediate_size,3072,


三个 `-` 行，就是 json 里三个不能照抄的旧名字：

| json 里写的 | 在 config 对象上 | 该怎么取 |
|---|---|---|
| `torch_dtype` | 废弃别名 | `config.dtype` |
| `rope_scaling` | 废弃别名 | `config.rope_parameters` |
| `rope_theta` | **取不到** | `config.rope_parameters['rope_theta']` |

前两个还能用，最后一个直接报 `AttributeError`。§6.1 的 RoPE 就得从
`rope_parameters` 字典里取 —— 凭 json 的印象写 `config.rope_theta`，代码跑不起来。

`+` 行多数不用管：那是 `PretrainedConfig` 基类给通用键备的默认值，框架的兜底，
不是 Qwen3 的选择。


### 0.4 维度常量

后面每一节都要用到的形状，全部从 `config` 对象读出来记成常量。
这样"复现结果和官方一致"才是真的，而不是把数字抄对了。

In [118]:
N_LAYERS = config.num_hidden_layers
HIDDEN = config.hidden_size
N_HEADS = config.num_attention_heads
N_KV_HEADS = config.num_key_value_heads
HEAD_DIM = config.head_dim
N_REP = N_HEADS // N_KV_HEADS
INTERMEDIATE = config.intermediate_size
VOCAB = config.vocab_size
RMS_EPS = config.rms_norm_eps
ROPE_THETA = config.rope_parameters['rope_theta']
SCALING = HEAD_DIM ** -0.5

CONSTANTS = [
    ('N_LAYERS',     N_LAYERS,     'Transformer Block 层数'),
    ('HIDDEN',       HIDDEN,       '残差流的宽度，每层进出都是这个数'),
    ('N_HEADS',      N_HEADS,      'query 头数'),
    ('N_KV_HEADS',   N_KV_HEADS,   'key/value 头数'),
    ('N_REP',        N_REP,        '每个 kv 头被几个 q 头共享'),
    ('HEAD_DIM',     HEAD_DIM,     '单头宽度，config 写死，不是 HIDDEN / N_HEADS'),
    ('INTERMEDIATE', INTERMEDIATE, 'MLP 中间层宽度'),
    ('VOCAB',        VOCAB,        '词表大小'),
    ('RMS_EPS',      RMS_EPS,      'RMSNorm 的 eps'),
    ('ROPE_THETA',   ROPE_THETA,   'RoPE 频率基数'),
    ('SCALING',      SCALING,      'attention 缩放系数 = HEAD_DIM ** -0.5'),
]

show(CONSTANTS, header=['常量', '值', '含义'], fmt='{:.10g}')

常量,值,含义
N_LAYERS,28,Transformer Block 层数
HIDDEN,1024,残差流的宽度，每层进出都是这个数
N_HEADS,16,query 头数
N_KV_HEADS,8,key/value 头数
N_REP,2,每个 kv 头被几个 q 头共享
HEAD_DIM,128,单头宽度，config 写死，不是 HIDDEN / N_HEADS
INTERMEDIATE,3072,MLP 中间层宽度
VOCAB,151936,词表大小
RMS_EPS,1e-06,RMSNorm 的 eps
ROPE_THETA,1000000,RoPE 频率基数


### 0.5 证据链：本实验涉及的源码位置

下面的行号从实际加载的模块动态读取，不是手抄的。后续每个模块解剖时会再次引用。

In [119]:
show_evidence()


对象,源码位置
Qwen3RMSNorm,modeling_qwen3.py:49-67
└ forward,modeling_qwen3.py:59-64
Qwen3MLP,modeling_qwen3.py:70-83
└ forward,modeling_qwen3.py:81-83
Qwen3RotaryEmbedding,modeling_qwen3.py:86-137
└ forward,modeling_qwen3.py:124-137
rotate_half,modeling_qwen3.py:140-144
apply_rotary_pos_emb,modeling_qwen3.py:147-170
repeat_kv,modeling_qwen3.py:173-182
eager_attention_forward,modeling_qwen3.py:185-207


## 1. 入口：tokenizer 只负责把文字变成 input_ids

```text
自然语言
   ↓
tokenizer          ← 本实验不解剖它的内部
   ↓
input_ids          ← 解剖从这里开始
   ↓
Qwen3 前向传播
```

选这句话作为实验输入，是因为它在上下文里给出了 `X 是 Y 的首都` 的模式，
模型只要沿模式补全就应该预测出 `首都`。序列短（9 个 token），
后面 9×9 的 attention 矩阵可以整个打印出来看。

In [120]:
PROMPT = '北京是中国的首都，巴黎是法国的'

encoded = tokenizer(PROMPT, return_tensors='pt')
input_ids = encoded.input_ids.to(DEVICE)
BATCH, SEQ = input_ids.shape

show([('原文', PROMPT),
      ('input_ids shape', f'{tuple(input_ids.shape)}   (B={BATCH}, S={SEQ})'),
      ('input_ids', str(input_ids[0].tolist()))])

show([(position, token_id, repr(tokenizer.decode([token_id])),
       tokenizer.convert_ids_to_tokens([token_id])[0])
      for position, token_id in enumerate(input_ids[0].tolist())],
     header=['pos', 'id', 'decode', '字节形态'])

原文,北京是中国的首都，巴黎是法国的
input_ids shape,"(1, 9) (B=1, S=9)"
input_ids,"[68990, 105196, 9370, 106114, 3837, 106004, 20412, 104328, 9370]"


pos,id,decode,字节形态
0,68990,'北京',åĮĹäº¬
1,105196,'是中国',æĺ¯ä¸ŃåĽ½
2,9370,'的',çļĦ
3,106114,'首都',é¦ĸéĥ½
4,3837,'，',ï¼Į
5,106004,'巴黎',å·´é»İ
6,20412,'是',æĺ¯
7,104328,'法国',æ³ķåĽ½
8,9370,'的',çļĦ


上面 `token` 列出现的 `åĮĹäº¬` 这类乱码是 byte-level BPE 的字节表示形式，`decode` 列才是可读文本。
这属于 tokenizer 内部机制，本实验不展开。我们只需要 `input_ids` 这 9 个整数。

# 第一遍：准备对照数据

这一遍不写任何自己的计算，只做一件事：把官方 forward 每个节点的中间状态**原样记下来**，
作为第二遍的比对基准。所以这两节全是取数的工具 —— 两份目录，两个动作。


## 2. 建两份目录：中间状态与模型参数

一次 forward 里能看的东西分两类：

| | 是什么 | 有多少 | 从哪来 |
|---|---|---|---|
| **中间状态** | 数据流过时每一步的样子 | 每层十几个，28 层几百个 | 只在 forward 时存在，跑完就没了 |
| **模型参数** | 训练完就固定的权重矩阵 | 310 个张量 | 存在模型里，不跑也能看 |

复现一个节点要同时用到两边：拿**参数**去算，拿**中间状态**当答案对照。
所以各建一份目录 —— `qwen` 装中间状态，`W` 装参数。

**忘了有哪些字段，敲对象名回车就行**：`qwen`、`qwen.L[0]`、`W`、`W.L[0]` 都会打印自己的清单。


### 2.1 中间状态：`observe`

中间状态只在 forward 执行的那一瞬间存在。要留下它们，得在每个子模块上挂 forward hook，
跑一次，再把 hook 摘掉。`observe` 把这三步包成一句：

```python
qwen = observe(model, input_ids)     # 跑一次官方 forward，记下全部内部状态
qwen.L[0].q_proj                     # 取第 0 层 q_proj 的输出，能 Tab 补全
```


In [121]:
qwen = observe(model, input_ids)
qwen

字段,形状,是什么
input_ids,"(1, 9)",输入 token id
embed,"(1, 9, 1024)",embed_tokens 查表，进第 0 层之前
L[0] … L[27],"(1, 9, 1024)",28 层 Decoder Layer，每层保形
final_norm,"(1, 9, 1024)",model.norm，最后一次 RMSNorm
logits,"(1, 9, 151936)",lm_head 投到 151936 词表
要做什么,怎么敲,说明
看某一层,qwen.L[0],直接敲，会打印字段清单
跨层共享件,qwen.cos qwen.sin qwen.mask qwen.position_ids,
看某个张量,look(qwen.L[0].q_proj),§3 的两个动作之一
比对结果,"check(我算的, qwen.L[0].q_proj)",§3 的两个动作之一


In [122]:
qwen.L[0]

字段（可 Tab 补全）,形状,是什么
.inp,"(1, 9, 1024)",这一层的输入（未归一化，残差记住的就是它）
.norm1,"(1, 9, 1024)",input_layernorm，pre-norm
.q_proj,"(1, 9, 2048)",1024 → 2048，16 头 × 128
.k_proj,"(1, 9, 1024)",1024 → 1024，8 头 × 128
.v_proj,"(1, 9, 1024)",1024 → 1024，8 头 × 128
.q_norm,"(1, 9, 16, 128)",在 head_dim=128 上归一化
.k_norm,"(1, 9, 8, 128)",同上。V 没有 norm
.attn_weights,"(1, 16, 9, 9)",softmax 之后的注意力权重
.o_proj,"(1, 9, 1024)",2048 → 1024
.attn_out,"(1, 9, 1024)",Attention 的最终输出（= o_proj 的输出）


### 2.2 模型参数：权重目录 `W`

参数不用跑就在模型里，裸路径 `model.model.layers[0].self_attn.q_proj.weight` 也能取。
建目录只为省事：不用记有哪些字段、也不用记形状。

```python
W = params(model)                   # 建目录，只调一次
W.L[0].q_proj                       # 取第 0 层 q_proj 的权重
W.find('norm')                      # 忘了字段全名时模糊搜
```

**两份目录的字段名是同一套**，同一个 `q_proj`：

| | 取到什么 | 形状 |
|---|---|---|
| `qwen.L[0].q_proj` | 那一层 q_proj 的**输出** | `(1, 9, 2048)` |
| `W.L[0].q_proj` | 它的**权重** | `(2048, 1024)` |

字段名或层号写错时，报错会把有效的那些列出来，不用回头翻。


In [123]:
W = params(model)
W

字段,形状,裸路径,怎么用
W.embed,"(151936, 1024)",model.model.embed_tokens.weight,查表取行 → my_embedding
W.final_norm,"(1024,)",model.model.norm.weight,"my_rmsnorm(x, w)"
W.lm_head,"(151936, 1024)",model.lm_head.weight,"my_linear(x, w) <- 与 embed 同一块内存"
W.inv_freq,"(64,)",model.model.rotary_emb.inv_freq,buffer，不是 parameter


In [124]:
W.L[0]

字段,形状,怎么用,裸路径（可直接复制）
.norm1,"(1024,)","my_rmsnorm(x, w)",input_layernorm.weight
.q_proj,"(2048, 1024)","my_linear(x, w)",self_attn.q_proj.weight
.k_proj,"(1024, 1024)","my_linear(x, w)",self_attn.k_proj.weight
.v_proj,"(1024, 1024)","my_linear(x, w)",self_attn.v_proj.weight
.q_norm,"(128,)","my_rmsnorm(x, w)",self_attn.q_norm.weight
.k_norm,"(128,)","my_rmsnorm(x, w)",self_attn.k_norm.weight
.o_proj,"(1024, 2048)","my_linear(x, w)",self_attn.o_proj.weight
.norm2,"(1024,)","my_rmsnorm(x, w)",post_attention_layernorm.weight
.gate,"(3072, 1024)","my_linear(x, w)",mlp.gate_proj.weight
.up,"(3072, 1024)","my_linear(x, w)",mlp.up_proj.weight


In [125]:
# 只记得名字里带 norm，不确定完整字段名
W.find('norm')

字段,形状,裸路径
W.final_norm,"(1024,)",model.model.norm.weight
W.L[i].norm1,"(1024,)",...layers[i].input_layernorm.weight
W.L[i].q_norm,"(128,)",...layers[i].self_attn.q_norm.weight
W.L[i].k_norm,"(128,)",...layers[i].self_attn.k_norm.weight
W.L[i].norm2,"(1024,)",...layers[i].post_attention_layernorm.weight


## 3. 两个动作：`look` 看，`check` 比

两份目录建好了，剩下两件事：把张量看清楚，判断自己算的对不对。

**`look(任何东西)`** 自己判断该怎么显示 —— 整型张量逐 token 解码，注意力方阵画成
token × token 网格，权重矩阵取左上角一块。完整张量始终在，只是不全打印；
想换 head 或位置再看，补个关键字参数（`look(w, head=5)`）。

**`check(我的, 官方的)`** 判据固定为 `max|差| / max|ref| < 1e-5`，**不留公差参数** ——
留了口子就有「调松一点让它过」的余地。每次调用记进全局清单，`summary()` 一次性汇总，
清单顺序就是调用顺序，哪个节点先出错不用另找。


In [126]:
# look 认中间状态：形状和类型不同，显示方式跟着变。
look(input_ids, 'input_ids')                    # 整型 → 逐 token 解码
look(qwen.L[0].out, 'Layer 0 输出')             # 三维 → 形状 + 一个数值窗口
look(qwen.L[0].attn_weights, 'Layer 0 head 0')  # 方阵注意力 → token × token 网格


pos,id,decode
0,68990,'北京'
1,105196,'是中国'
2,9370,'的'
3,106114,'首都'
4,3837,'，'
5,106004,'巴黎'
6,20412,'是'
7,104328,'法国'
8,9370,'的'


形状,"(1, 9, 1024)"
"[0, -1, :6]","[-0.9747, -0.4370, +0.0096, -0.6557, +0.1166, +0.1212]"


query,k0,k1,k2,k3,k4,k5,k6,k7,k8,query 的 token
0,100.0,–,–,–,–,–,–,–,–,'北京'
1,44.7,55.3,–,–,–,–,–,–,–,'是中国'
2,3.8,68.6,27.5,–,–,–,–,–,–,'的'
3,39.0,3.8,13.4,43.8,–,–,–,–,–,'首都'
4,0.7,5.6,13.4,0.2,80.1,–,–,–,–,'，'
5,0.7,2.6,41.6,8.8,37.1,9.2,–,–,–,'巴黎'
6,0.4,1.4,35.5,0.2,49.7,1.5,11.3,–,–,'是'
7,2.2,2.5,10.3,5.1,20.6,42.7,12.3,4.4,–,'法国'
8,1.6,10.6,21.2,0.6,22.5,0.5,16.9,0.4,25.8,'的'


In [127]:
# look 也认权重：目录的清单只给形状，实际数值交给 look。
look(W.L[0].q_proj, 'q_proj 权重')              # 二维 → 左上角一小块，注明 [out, in]
look(W.L[0].norm1, 'input_layernorm 权重')      # 一维 → 开头几个值


形状,"(2048, 1024) 矩阵 [out=2048, in=1024]"
怎么用,"my_linear(x, w) 内部做 x @ w.T，不必手动转置"
"w[0, :6]","[+0.0059, -0.0042, -0.0137, +0.0197, +0.0142, -0.0013] ← 第 0 个输出通道，对应前 6 个输入维"
"w[1, :6]","[-0.0264, +0.0089, -0.0012, +0.0220, +0.0011, +0.0027]"
"w[2, :6]","[+0.0220, +0.0032, -0.0033, +0.0091, +0.0028, -0.0067]"
分布,mean=-0.0000 std=0.0317 absmax=0.6602


形状,"(1024,) 向量 [dim=1024]"
怎么用,"my_rmsnorm(x, w) 内部逐元素相乘，不改形状"
w[:6],"[+0.1377, +0.7109, +0.5977, +0.7227, +0.2070, +0.6367]"
分布,mean=+0.1773 std=0.0753 absmax=1.0469


In [128]:
# 两边都是官方张量，答案已知：Layer 0 的输出就是 Layer 1 的输入，中间没有别的运算。
# 拿它先看看 check 通过时长什么样 —— 真正的复现比对从 §4 开始。
check(qwen.L[0].out, qwen.L[1].inp, '官方 L0 输出直连 L1 输入');

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  官方 L0 输出直连 L1 输入


# 第二遍：自己复现

原则是**能自己实现就自己实现**，但不重复制造 PyTorch。

**自己实现**：Embedding、Linear、RMSNorm、RoPE、GQA、Attention、causal mask、MLP、Residual、Decoder Layer。

**直接用 PyTorch 基础算子**：`matmul`、`reshape`、`view`、`transpose`、`softmax`、`exp`、`rsqrt`、`sigmoid` 等。

Linear 和 RMSNorm 是第一次完整实现，之后所有地方复用自己的这一份实现。

## 4. 最基础的三块积木

后面每一个节点都由三个函数拼出来：`my_linear`、`my_rmsnorm`、`my_silu`。
它们不含任何 Qwen3 特有的结构，所以先在这里逐个跟官方算子对齐 ——
之后哪一步对不上，就能确定问题不在积木本身。


### 4.1 Linear（无 bias）

`config.attention_bias = False`，且 MLP 的三个投影在源码里都是 `bias=False`，
所以本模型**所有** Linear 都没有 bias。计算就是一次矩阵乘：

$$\mathrm{Linear}(x) = x W^\top$$

`nn.Linear` 的权重形状是 `[out_features, in_features]`，所以要转置后右乘。

In [129]:
def my_linear(x, weight):
    """无 bias 的线性层。weight: [out_features, in_features]"""
    assert weight.dim() == 2, (
        f'my_linear 的第二个参数要是二维权重矩阵，收到 {weight.dim()} 维 '
        f'{tuple(weight.shape)}。一维的是 RMSNorm 系数，该用 my_rmsnorm')
    assert x.shape[-1] == weight.shape[1], (
        f'形状对不上：x 最后一维 {x.shape[-1]}，而 weight 的 in_features 是 '
        f'{weight.shape[1]}（weight 形状 {tuple(weight.shape)} 是 [out, in]）')
    return x @ weight.T


# 第一次用权重目录当参数：W.L[0].q_proj 就是 model.model.layers[0].self_attn.q_proj.weight。
check(my_linear(qwen.L[0].norm1, W.L[0].q_proj), qwen.L[0].q_proj, 'my_linear vs q_proj');

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_linear vs q_proj


### 4.2 RMSNorm

源码 `Qwen3RMSNorm.forward`（见 §0.5 证据链）的计算顺序是：

$$\bar{x} = x \cdot \frac{1}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}, \qquad
\mathrm{RMSNorm}(x) = w \odot \bar{x}$$

严格按代码顺序，有四个容易写错的细节：

1. **先转 float32 再算**，最后才转回输入 dtype；
2. `weight` 的乘法发生在**转回 dtype 之后**，不是在 float32 里乘；
3. 用 `rsqrt(variance + eps)`，eps 在**根号内部**，不是外部；
4. 这里的 variance 是**平方的均值**，不减均值（这是 RMSNorm 与 LayerNorm 的区别）。

In [130]:
def my_rmsnorm(x, weight, eps=RMS_EPS):
    """严格按 Qwen3RMSNorm.forward 的顺序实现。归一化沿最后一维进行。"""
    assert weight.dim() == 1, (
        f'my_rmsnorm 的第二个参数要是一维缩放系数，收到 {weight.dim()} 维 '
        f'{tuple(weight.shape)}。\n参数顺序是 my_rmsnorm(输入, 权重)')
    assert x.shape[-1] == weight.shape[0], (
        f'形状对不上：x 最后一维 {x.shape[-1]}，权重长度 {weight.shape[0]}')
    input_dtype = x.dtype
    x = x.to(torch.float32)                              # 1. 升到 float32
    variance = x.pow(2).mean(-1, keepdim=True)           # 2. 平方的均值，不减均值
    x = x * torch.rsqrt(variance + eps)                  # 3. eps 在根号内
    return weight * x.to(input_dtype)                    # 4. 先转回 dtype，再乘 weight


check(my_rmsnorm(qwen.L[0].inp, W.L[0].norm1),
      qwen.L[0].norm1, 'my_rmsnorm vs input_layernorm')
# post_attention_layernorm 的输入是"第一次残差之后"的中间态。
# 它不是任何模块的直接输出，工具已经替我们拼好了：qwen.L[0].after_attn = inp + attn_out。
check(my_rmsnorm(qwen.L[0].after_attn, W.L[0].norm2),
      qwen.L[0].norm2, 'my_rmsnorm vs post_attn_norm');

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_rmsnorm vs input_layernorm
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_rmsnorm vs post_attn_norm


In [131]:
# 这两个参数都是张量，传反了形状依然能广播，所以要靠断言拦。
print('参数传反 my_rmsnorm(权重, 输入) 会怎样：')
try:
    my_rmsnorm(W.L[0].norm1, qwen.L[0].inp)
    print('  居然没报错 ← 断言失效了')
except AssertionError as e:
    for _i, _line in enumerate(str(e).splitlines()):
        print(('  断言拦住了：' if _i == 0 else '  　　　　　　') + _line.strip())
print('  没有断言的话，(1024,) 与 (1, 9, 1024) 会广播成功，形状对、不报错，')
print('  但归一化的对象变成了权重，输入反而成了缩放系数。')

参数传反 my_rmsnorm(权重, 输入) 会怎样：
  断言拦住了：my_rmsnorm 的第二个参数要是一维缩放系数，收到 3 维 (1, 9, 1024)。
  　　　　　　参数顺序是 my_rmsnorm(输入, 权重)
  没有断言的话，(1024,) 与 (1, 9, 1024) 会广播成功，形状对、不报错，
  但归一化的对象变成了权重，输入反而成了缩放系数。


### 4.3 SiLU 激活

`config.hidden_act = 'silu'`，即 $\mathrm{SiLU}(x) = x \cdot \sigma(x) = \dfrac{x}{1 + e^{-x}}$。
我们用 `torch.sigmoid` 这个基础算子拼出来，不调 `nn.functional.silu`。

In [132]:
def my_silu(x):
    return x * torch.sigmoid(x)


_t = torch.randn(4, 8, dtype=DTYPE)
check(my_silu(_t), torch.nn.functional.silu(_t), 'my_silu vs F.silu');

✓  max_abs=1.192e-07  mean_abs=9.662e-09  max_rel=6.356e-08  my_silu vs F.silu


## 5. Embedding：一次按行查表

```text
              input_ids [B, S]              整数，取值范围 [0, 151936)
                   │
                   │  以 token id 作为行号，从权重矩阵取行
                   ▼
   embed_tokens.weight [151936, 1024]
                   │
                   ▼
        inputs_embeds [B, S, 1024]
```

`nn.Embedding` 的 forward 没有矩阵乘、没有激活，就是一次索引。所以"自己实现"它等价于
`weight[input_ids]`。本节末尾还要验证一件事：**这张表就是最后 lm_head 用的那张表**。

In [133]:
def my_embedding(ids, weight):
    """按行号取行。等价于 nn.Embedding.forward（无 padding_idx 参与时）。"""
    return weight[ids]


EMBED_WEIGHT = W.embed          # = model.model.embed_tokens.weight
my_embeds = my_embedding(input_ids, EMBED_WEIGHT)

check(my_embeds, qwen.embed, 'my_embedding vs embed_tokens')
check(my_embeds, qwen.L[0].inp, 'my_embedding vs Layer 0 输入');

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_embedding vs embed_tokens
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_embedding vs Layer 0 输入


In [134]:
# 逐个 token 确认"查表"就是字面意义的取行
_rows = []
for position in [0, SEQ - 1]:
    token_id = input_ids[0, position].item()
    row = EMBED_WEIGHT[token_id]
    _rows.append((
        position, token_id, repr(tokenizer.decode([token_id])),
        torch.equal(row, my_embeds[0, position]),
        '[' + ', '.join(f'{v:+.5f}' for v in row[:6].tolist()) + ', …]'))

show(_rows, header=['pos', 'id', 'token',
                    'embeds[0,pos] == weight[id]', 'weight[id] 的前 6 维'])

pos,id,token,"embeds[0,pos] == weight[id]",weight[id] 的前 6 维
0,68990,'北京',✓,"[-0.03198, -0.04858, +0.03345, +0.04492, -0.03296, -0.04077, …]"
8,9370,'的',✓,"[-0.01941, -0.00482, -0.04858, -0.01636, -0.00824, +0.02405, …]"


`tie_word_embeddings: true` 的含义在这里变得具体：**同一张 `[151936, 1024]` 的矩阵被用了两次**。

- 入口（Embedding）当字典查：行号 → 向量；
- 出口（LM Head）当打分器用：把 hidden state 和 151936 行逐行做内积。

这也是为什么 `model.safetensors` 里存了 310 个张量，却**没有** `lm_head.weight`——它根本不需要存。
这一张表占了模型总参数的四分之一以上。

In [135]:
# 要证明的是官方模型里 tie_word_embeddings 生效，所以右边一律用官方裸路径，
# 不拿目录自己的两个字段互证（那只能说明 params() 写对了）。
_official_lm_head = model.lm_head.weight
print(f'W.embed    (embed_tokens.weight)  {tuple(EMBED_WEIGHT.shape)}')
print(f'W.lm_head  (lm_head.weight)       {tuple(W.lm_head.shape)}')
print(f'W.lm_head 取到的就是它:  {W.lm_head is _official_lm_head}')
print(f'是同一个张量对象:       {EMBED_WEIGHT is _official_lm_head}')
print(f'共享同一块内存:         {EMBED_WEIGHT.data_ptr() == _official_lm_head.data_ptr()}')
print(f'_tied_weights_keys:    {type(model)._tied_weights_keys}')
print()
_embed_params = VOCAB * HIDDEN
_total = sum(p.numel() for p in model.parameters())
print(f'这张表的参数量: {_embed_params:,} = 总参数 {_total:,} 的 {_embed_params / _total:.1%}')

# 磁盘上到底存了什么：共享的那张表只以 embedding 的名字存一份。
with safe_open(MODEL_PATH / 'model.safetensors', framework='pt') as f:
    _keys = list(f.keys())
    _file_params = sum(prod(f.get_slice(k).get_shape()) for k in _keys)

print()
print(f'权重文件键数:   {len(_keys)}')
print(f"含 'lm_head' 的键: {[k for k in _keys if 'lm_head' in k]}")
print(f"含 'embed' 的键:   {[k for k in _keys if 'embed' in k]}")
print(f'文件内参数量:   {_file_params:,}  '
      f'(与 model.parameters() 计数相等: {_file_params == _total})')


W.embed    (embed_tokens.weight)  (151936, 1024)
W.lm_head  (lm_head.weight)       (151936, 1024)
W.lm_head 取到的就是它:  True
是同一个张量对象:       True
共享同一块内存:         True
_tied_weights_keys:    {'lm_head.weight': 'model.embed_tokens.weight'}

这张表的参数量: 155,582,464 = 总参数 596,049,920 的 26.1%

权重文件键数:   310
含 'lm_head' 的键: []
含 'embed' 的键:   ['model.embed_tokens.weight']
文件内参数量:   596,049,920  (与 model.parameters() 计数相等: True)


## 6. 进入 Layer 之前：两样全局预备件

看源码 `Qwen3Model.forward` 会发现，进入 28 层循环之前先算好了两样东西，
然后**原样传给每一层**，28 层共用，不重复计算：

```text
inputs_embeds
     │
     ├──→ position_ids ──→ rotary_emb ──→ (cos, sin)   ─┐
     │                                                  ├─→ 传给全部 28 层
     └──→ create_causal_mask ──→ attention_mask        ─┘
```

这一点很容易被忽略：RoPE 的 cos/sin **不在 Attention 内部计算**，而是模型级别算一次。

### 6.1 RoPE：cos / sin 表

RoPE 的目标是把"位置"编码成一个旋转。它要两样输入：位置序号，和一组频率。
位置序号就是 `0..S-1`（`Qwen3Model.forward` 里没传 `position_ids` 时的默认值）；
频率向量（`inv_freq`）只依赖 head_dim 和 theta：

$$\theta_j = \frac{1}{\text{base}^{\,2j/d}}, \qquad j = 0, 1, \dots, \frac{d}{2}-1$$

其中 $d = 128$（head_dim），base = `rope_theta` = 1000000。注意源码里
`torch.arange(0, dim, 2) / dim` 得到的是 $2j/d$，所以 `inv_freq` 长度是 64。

然后与位置做外积，再把结果**复制一份拼接**成 128 维：

$$\text{freqs}[p, j] = p \cdot \theta_j \quad (\text{形状 } S \times 64), \qquad
\text{emb} = [\text{freqs}, \text{freqs}] \quad (S \times 128)$$

$$\cos = \cos(\text{emb}), \qquad \sin = \sin(\text{emb})$$

拼接这一步是为了配合后面 `rotate_half` 的实现方式。

In [136]:
# Qwen3Model.forward 里的 position_ids：没传就是 0..S-1
my_position_ids = torch.arange(SEQ, device=DEVICE).unsqueeze(0)
print(f'position_ids {tuple(my_position_ids.shape)} = {my_position_ids[0].tolist()}')

# 官方实际用的 position_ids（工具已从 Decoder Layer 的 kwargs 抓好）
check(my_position_ids.float(), qwen.position_ids.float(), 'my_position_ids');

position_ids (1, 9) = [0, 1, 2, 3, 4, 5, 6, 7, 8]
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_position_ids


In [137]:
def my_rope_tables(position_ids, head_dim=HEAD_DIM, base=ROPE_THETA, dtype=DTYPE):
    """复现 Qwen3RotaryEmbedding：返回 (cos, sin)，形状 [B, S, head_dim]。"""
    # inv_freq: [head_dim/2]，全程 float32
    exponent = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
    inv_freq = 1.0 / (base ** exponent)

    # 外积：[B, head_dim/2, 1] @ [B, 1, S] -> [B, head_dim/2, S] -> transpose -> [B, S, head_dim/2]
    inv_freq_expanded = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
    positions_expanded = position_ids[:, None, :].float()
    freqs = (inv_freq_expanded @ positions_expanded).transpose(1, 2)

    emb = torch.cat((freqs, freqs), dim=-1)      # [B, S, head_dim]
    return emb.cos().to(dtype), emb.sin().to(dtype), inv_freq


my_cos, my_sin, my_inv_freq = my_rope_tables(my_position_ids)

print(f'inv_freq {tuple(my_inv_freq.shape)}  前4 = '
      f'[{", ".join(f"{v:.3e}" for v in my_inv_freq[:4].tolist())}]')
print(f'         后4 = [{", ".join(f"{v:.3e}" for v in my_inv_freq[-4:].tolist())}]')
print(f'cos/sin  {tuple(my_cos.shape)}')
print()
check(my_cos, qwen.cos, 'my_rope cos')
check(my_sin, qwen.sin, 'my_rope sin')
check(my_inv_freq, W.inv_freq, 'my_inv_freq');

inv_freq (64,)  前4 = [1.000e+00, 8.058e-01, 6.494e-01, 5.233e-01]
         后4 = [2.371e-06, 1.911e-06, 1.540e-06, 1.241e-06]
cos/sin  (1, 9, 128)

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_rope cos
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_rope sin
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_inv_freq


In [138]:
# 观察窗口：cos 表的前 4 个位置 × 前 3 个频率通道
show([(f'pos {p}', *[my_cos[0, p, c].item() for c in range(3)]) for p in range(4)],
     header=['位置', 'ch0', 'ch1', 'ch2'], fmt='{:+.6f}',
     title='cos[0, position, channel] 的一角')

# ch0 频率最高（相邻位置差异大），高编号通道频率极低（长距离才有区分度）
show([(f'ch{channel}', *[my_cos[0, p, channel].item() for p in range(SEQ)])
      for channel in [0, 32, 63]],
     header=['通道', *[f'pos{p}' for p in range(SEQ)]], fmt='{:+.4f}',
     title='同一通道沿位置怎么变：ch0 快，ch63 几乎不动')

位置,ch0,ch1,ch2
pos 0,+1.000000,+1.000000,+1.000000
pos 1,+0.540302,+0.692504,+0.796458
pos 2,-0.416147,-0.040877,+0.268690
pos 3,-0.989992,-0.749119,-0.368457


通道,pos0,pos1,pos2,pos3,pos4,pos5,pos6,pos7,pos8
ch0,+1.0000,+0.5403,-0.4161,-0.9900,-0.6536,+0.2837,+0.9602,+0.7539,-0.1455
ch32,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000
ch63,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000,+1.0000


### 6.2 Causal Mask

因果掩码保证位置 $i$ 只能看到 $j \le i$ 的位置。eager 路径下它是一个**加性** mask：
允许的位置填 0，禁止的位置填一个极小的数（`torch.finfo(float32).min`），
加到 attention 分数上之后，softmax 会把这些位置压到 0。

```text
        j=0   1    2   ...          ← 被看的位置（key）
 i=0  [  0  -inf -inf ...  ]
 i=1  [  0    0  -inf ...  ]        ← 看的位置（query）
 i=2  [  0    0    0  ...  ]
```

In [139]:
def my_causal_mask(seq_len, dtype=DTYPE, device=DEVICE):
    """加性因果掩码，形状 [1, 1, S, S]，可广播到 [B, heads, S, S]。"""
    blocked = torch.finfo(dtype).min
    positions = torch.arange(seq_len, device=device)
    # query 位置 i 只能看 key 位置 j <= i
    allowed = positions[:, None] >= positions[None, :]
    mask = torch.where(allowed, torch.zeros((), dtype=dtype, device=device),
                       torch.full((), blocked, dtype=dtype, device=device))
    return mask[None, None, :, :]


my_mask = my_causal_mask(SEQ)

print(f'官方 mask: {tuple(qwen.mask.shape)}  dtype={qwen.mask.dtype}')
print(f'屏蔽值:    {qwen.mask.min().item():.6e}')
print(f'等于 torch.finfo(float32).min: '
      f'{qwen.mask.min().item() == torch.finfo(torch.float32).min}')
print()
check(my_mask, qwen.mask, 'my_causal_mask')
print()
print('mask[0,0] 的 0/-inf 结构（0 表示可见，. 表示屏蔽）：')
for i in range(SEQ):
    row = ' '.join('0' if qwen.mask[0, 0, i, j] == 0 else '.' for j in range(SEQ))
    print(f'  i={i}  {row}')

官方 mask: (1, 1, 9, 9)  dtype=torch.float32
屏蔽值:    -3.402823e+38
等于 torch.finfo(float32).min: True

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_causal_mask

mask[0,0] 的 0/-inf 结构（0 表示可见，. 表示屏蔽）：
  i=0  0 . . . . . . . .
  i=1  0 0 . . . . . . .
  i=2  0 0 0 . . . . . .
  i=3  0 0 0 0 . . . . .
  i=4  0 0 0 0 0 . . . .
  i=5  0 0 0 0 0 0 . . .
  i=6  0 0 0 0 0 0 0 . .
  i=7  0 0 0 0 0 0 0 0 .
  i=8  0 0 0 0 0 0 0 0 0


### 6.3 两样预备件都是 28 层共用

本章开头说这两样"算一次然后原样传给每一层"。这里直接比对象身份：
不是数值相等，是同一个 Python 对象 —— 28 层里没有任何一层重新算过。

这个结论不是逐元素比对，没有误差可言，所以走 `record` 而不是 `check`，
一样进 `summary()` 的清单。误差那两栏打成灰短横，表示没测过。


In [140]:
mask_shared = all(qwen.L[i].mask is qwen.mask for i in range(N_LAYERS))
rope_shared = all(qwen.L[i].cos is qwen.cos for i in range(N_LAYERS))
print(f'28 层共用同一个 attention_mask 对象: {mask_shared}')
print(f'28 层共用同一组 (cos, sin) 对象:      {rope_shared}')
record('28 层共用 mask 与 RoPE 表', mask_shared and rope_shared);

28 层共用同一个 attention_mask 对象: True
28 层共用同一组 (cos, sin) 对象:      True


## 7. 完整解剖 Layer 0

28 层结构完全相同，所以只对第 0 层做逐步解剖。后续 27 层用同样的逻辑批量计算并全部验证，
但不重复写 27 遍说明。

```text
                           Layer input
                                │
             ┌──────────────────┤ residual
             │                  ▼
             │        input_layernorm (RMSNorm)
             │                  ▼
             │      ┌────────────────────────┐
             │      │                        │
             │      │       Attention        │
             │      │                        │
             │      └────────────────────────┘
             │                  ▼
             └────────────────► ⊕  residual add
                                │
             ┌──────────────────┤ residual
             │                  ▼
             │        post_attention_layernorm
             │                  ▼
             │      ┌────────────────────────┐
             │      │                        │
             │      │          MLP           │
             │      │                        │
             │      └────────────────────────┘
             │                  ▼
             └────────────────► ⊕  residual add
                                │
                           Layer output
```

源码依据：`Qwen3DecoderLayer.forward`（§0.5 证据链给出行号）。
注意两个 RMSNorm 都在**子模块之前**（pre-norm），残差加的是**未归一化**的输入。

### 7.1 Layer 输入与 input_layernorm

先取这一层的官方中间状态和权重，再走第一个 pre-norm。

`residual_1` 记住的是**未归一化**的 layer 输入 —— 归一化后的 `h` 只喂给 Attention，
残差那条路加回来的是 `x_in` 本身。§7.8 会用到这个区别。

In [141]:
LAYER = 0
L0 = qwen.L[LAYER]          # 这一层的官方中间状态，敲 L0 回车看字段清单
P0 = W.L[LAYER]             # 这一层的 11 个权重，敲 P0 回车看字段清单
# 两边字段同名：L0.q_proj 是输出，P0.q_proj 是权重

# ── Layer 输入 ──
x_in = L0.inp               # = embedding 的输出，即实验 1 骨架表里 layers.0 的输入
residual_1 = x_in           # 残差记住的是未归一化的输入

h = my_rmsnorm(x_in, P0.norm1)
check(h, L0.norm1, f'L{LAYER} input_layernorm')

look(x_in, 'layer input')
look(h, 'after input_layernorm')


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 input_layernorm


形状,"(1, 9, 1024)"
"[0, -1, :6]","[-0.0194, -0.0048, -0.0486, -0.0164, -0.0082, +0.0240]"


形状,"(1, 9, 1024)"
"[0, -1, :6]","[-0.1020, -0.1308, -1.1080, -0.4511, -0.0651, +0.5843]"


### 7.2 Q / K / V 投影

三个投影的输出维度不一样，这是 GQA（Grouped Query Attention）的起点：

| 投影 | weight 形状 | 输出 | 含义 |
|---|---|---|---|
| `q_proj` | `[2048, 1024]` | `[B, S, 2048]` | 16 个 query 头 × 128 |
| `k_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 key 头 × 128 |
| `v_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 value 头 × 128 |

query 头数是 kv 头数的 2 倍。注意 `q_proj` 把 1024 维**升到了 2048**，
比 hidden_size 还大——这是 Qwen3 的选择，`head_dim=128` 是配置里写死的，不是 `hidden_size / n_heads`。

In [142]:
q_flat = my_linear(h, P0.q_proj)
k_flat = my_linear(h, P0.k_proj)
v_flat = my_linear(h, P0.v_proj)

check(q_flat, L0.q_proj, f'L{LAYER} q_proj')
check(k_flat, L0.k_proj, f'L{LAYER} k_proj')
check(v_flat, L0.v_proj, f'L{LAYER} v_proj');


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 q_proj
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 k_proj
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 v_proj


### 7.3 reshape 成多头，然后做 q_norm / k_norm

**这一步是 Qwen3 与 Llama 最关键的结构差异。** 源码里这三行把好几个操作压在了一起：

```python
query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
key_states   = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
```

拆开看，真实顺序是：

```text
q_proj 输出  [B, S, 2048]
    ↓ view(B, S, -1, 128)
             [B, S, 16, 128]
    ↓ q_norm  ← RMSNorm 作用在最后一维 head_dim=128 上，不是 1024！
             [B, S, 16, 128]
    ↓ transpose(1, 2)
             [B, 16, S, 128]
```

三个要点：

1. norm 在 **reshape 之后**做，所以归一化的单位是**每个头的 128 维向量**，不是整个 2048；
2. `q_norm.weight` 的形状是 `[128]`，**16 个头共享同一组 128 个缩放参数**；
3. **V 没有 norm**，只有 Q 和 K 有。

In [143]:
print(f'q_norm 权重形状: {tuple(P0.q_norm.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_HEADS} 个 q 头共享')
print(f'k_norm 权重形状: {tuple(P0.k_norm.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_KV_HEADS} 个 kv 头共享')
print(f'v 有 norm 吗: {hasattr(model.model.layers[LAYER].self_attn, "v_norm")}')
print()

# view: 把最后一维拆成 (头数, head_dim)
q_heads = q_flat.view(BATCH, SEQ, N_HEADS, HEAD_DIM)
k_heads = k_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)
v_heads = v_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)

# q_norm / k_norm 在 head_dim 上做 RMSNorm
q_normed = my_rmsnorm(q_heads, P0.q_norm)
k_normed = my_rmsnorm(k_heads, P0.k_norm)

check(q_normed, L0.q_norm, f'L{LAYER} q_norm')
check(k_normed, L0.k_norm, f'L{LAYER} k_norm');


q_norm 权重形状: (128,)  ← 长度 128，被 16 个 q 头共享
k_norm 权重形状: (128,)  ← 长度 128，被 8 个 kv 头共享
v 有 norm 吗: False

✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 q_norm
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 k_norm


In [144]:
# transpose 把头维提到前面，之后所有 attention 计算都在 [B, heads, S, head_dim] 上做
q = q_normed.transpose(1, 2)
k = k_normed.transpose(1, 2)
v = v_heads.transpose(1, 2)

print('norm 前后对比（head 0，最后一个位置，前 6 维）：')
look(q_heads.transpose(1, 2), 'q 未 norm')
look(q, 'q 已 norm')
print()
print('验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1')
_h0 = q_heads[0, -1, 0].float()
print(f'  norm 前 RMS = {_h0.pow(2).mean().sqrt().item():.4f}')
_h0n = (q_normed[0, -1, 0] / P0.q_norm).float()
print(f'  norm 后（除掉 weight）RMS = {_h0n.pow(2).mean().sqrt().item():.6f}')


norm 前后对比（head 0，最后一个位置，前 6 维）：


形状,"(1, 16, 9, 128)"
"[0, 0, -1, :6]","[+0.0292, +0.1119, -0.0286, -0.1662, -0.0711, +0.1085]"


形状,"(1, 16, 9, 128)"
"[0, 0, -1, :6]","[+0.5642, +0.5918, +0.0894, -1.2054, -0.7853, +0.7144]"



验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1
  norm 前 RMS = 0.2349
  norm 后（除掉 weight）RMS = 0.999991


### 7.4 应用 RoPE

$$q' = q \odot \cos + \mathrm{rotate\_half}(q) \odot \sin$$

其中 `rotate_half` 把 128 维**前后对半切开**再交叉取负：

$$\mathrm{rotate\_half}([x_1, x_2]) = [-x_2, x_1], \qquad x_1 = x[:64],\ x_2 = x[64:]$$

这与"把相邻两维配成一对做二维旋转"的经典写法在数学上等价，但**维度配对方式不同**：
这里配对的是 $(i, i+64)$，不是 $(2i, 2i+1)$。这也解释了 §6.1 为什么要把 freqs 复制拼接成 128 维——
`cos` 的第 $i$ 维和第 $i+64$ 维是同一个角度。

cos/sin 形状是 `[B, S, 128]`，要 `unsqueeze(1)` 变成 `[B, 1, S, 128]` 才能广播到 `[B, heads, S, 128]`。
**Q 和 K 都要转，V 不转。**

In [145]:
def my_rotate_half(x):
    """把最后一维对半切开，交叉取负。"""
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((-x2, x1), dim=-1)

def my_apply_rope(q, k, cos, sin):
    """cos/sin: [B, S, head_dim] -> unsqueeze 到 [B, 1, S, head_dim] 广播。"""
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_out = q * cos + my_rotate_half(q) * sin
    k_out = k * cos + my_rotate_half(k) * sin
    return q_out, k_out

check(my_rotate_half(q), Q3.rotate_half(q), 'my_rotate_half')

q_rope, k_rope = my_apply_rope(q, k, my_cos, my_sin)

_ref_q, _ref_k = Q3.apply_rotary_pos_emb(q, k, qwen.cos, qwen.sin)
check(q_rope, _ref_q, f'L{LAYER} RoPE(q)')
check(k_rope, _ref_k, f'L{LAYER} RoPE(k)');


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  my_rotate_half
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 RoPE(q)
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 RoPE(k)


In [146]:
# RoPE 保长度：旋转不改变向量的模
_before = q[0, 0, -1].float().norm().item()
_after = q_rope[0, 0, -1].float().norm().item()
print(f'RoPE 前后向量模长（head 0, 最后位置）: {_before:.6f} → {_after:.6f}')
print(f'相对变化: {abs(_after - _before) / _before:.2e}   ← 旋转是保长变换')
print()
print('位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：')
_p0_diff = (q_rope[0, 0, 0] - q[0, 0, 0]).abs().max().item()
print(f'  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = {_p0_diff:.3e}')

RoPE 前后向量模长（head 0, 最后位置）: 16.679739 → 16.679739
相对变化: 0.00e+00   ← 旋转是保长变换

位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：
  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = 0.000e+00


### 7.5 GQA：repeat_kv

16 个 query 头要和 8 个 kv 头对齐。做法是把每个 kv 头**复制 2 份**：

```text
k: [B, 8, S, 128]
     ↓ 插入一个长度 2 的维度并 expand
   [B, 8, 2, S, 128]
     ↓ reshape 合并前两维
   [B, 16, S, 128]
```

复制方式是 `repeat_interleave` 语义：kv 头 0 服务 q 头 0 和 1，kv 头 1 服务 q 头 2 和 3，以此类推。
`expand` 不复制内存，`reshape` 才实际展开。

这一步发生在 **RoPE 之后**。顺序很重要：如果先 repeat 再转 RoPE，就要多算一倍的旋转。

In [147]:
def my_repeat_kv(hidden_states, n_rep):
    """[B, KV, S, D] -> [B, KV*n_rep, S, D]，等价于 repeat_interleave(dim=1)。"""
    batch, n_kv, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    expanded = hidden_states[:, :, None, :, :].expand(batch, n_kv, n_rep, seq_len, head_dim)
    return expanded.reshape(batch, n_kv * n_rep, seq_len, head_dim)

k_rep = my_repeat_kv(k_rope, N_REP)
v_rep = my_repeat_kv(v, N_REP)

check(k_rep, Q3.repeat_kv(_ref_k, N_REP), f'L{LAYER} repeat_kv(k)')
check(v_rep, Q3.repeat_kv(v, N_REP), f'L{LAYER} repeat_kv(v)')

print()
print('确认复制的对应关系（q 头 i ← kv 头 i // 2）：')
for q_head in [0, 1, 2, 3, 14, 15]:
    kv_head = q_head // N_REP
    same = torch.equal(k_rep[0, q_head], k_rope[0, kv_head])
    print(f'  k_rep[head {q_head:2d}] == k_rope[head {kv_head}] : {same}')


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 repeat_kv(k)
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 repeat_kv(v)

确认复制的对应关系（q 头 i ← kv 头 i // 2）：
  k_rep[head  0] == k_rope[head 0] : True
  k_rep[head  1] == k_rope[head 0] : True
  k_rep[head  2] == k_rope[head 1] : True
  k_rep[head  3] == k_rope[head 1] : True
  k_rep[head 14] == k_rope[head 7] : True
  k_rep[head 15] == k_rope[head 7] : True


### 7.6 QKᵀ、mask、softmax

$$\text{scores} = \frac{Q K^\top}{\sqrt{d_k}}, \qquad d_k = 128,\ \frac{1}{\sqrt{128}} \approx 0.088388$$

$$A = \mathrm{softmax}(\text{scores} + \text{mask})$$

严格按源码 `eager_attention_forward`：

1. 缩放在 matmul **之后**乘（`torch.matmul(q, k.T) * scaling`），不是先缩放 q；
2. mask 是**加**上去的，不是乘或填充；
3. softmax 指定 `dtype=torch.float32`，算完再转回 q 的 dtype。

shape 变化是整个 Attention 里最需要盯住的一段：
`[B, 16, S, 128] @ [B, 16, 128, S] → [B, 16, S, S]`。序列维度出现了两次，
第一个 S 是"谁在看"，第二个 S 是"看谁"。

In [148]:
scores = torch.matmul(q_rope, k_rep.transpose(2, 3)) * SCALING
scores_masked = scores + my_mask
attn_weights = torch.softmax(scores_masked, dim=-1, dtype=torch.float32).to(q_rope.dtype)

check(attn_weights, L0.attn_weights, f'L{LAYER} attn_weights')

print()
print(f'每行和离 1 最远: '
      f'{(attn_weights.sum(-1).float() - 1).abs().max().item():.3e}   ← 应为 0')
print(f'被 mask 的位置权重最大值: {attn_weights[0, 0, 0, 1:].max().item():.3e}   ← 应为 0')


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 attn_weights

每行和离 1 最远: 2.384e-07   ← 应为 0
被 mask 的位置权重最大值: 0.000e+00   ← 应为 0


In [149]:
# 观察窗口：head 0 的完整 9×9 attention 矩阵。
# look 认得 [B, heads, S, S] 这个形状，直接画成 token × token 网格。
look(attn_weights, f'Layer {LAYER}, head 0')

query,k0,k1,k2,k3,k4,k5,k6,k7,k8,query 的 token
0,100.0,–,–,–,–,–,–,–,–,'北京'
1,44.7,55.3,–,–,–,–,–,–,–,'是中国'
2,3.8,68.6,27.5,–,–,–,–,–,–,'的'
3,39.0,3.8,13.4,43.8,–,–,–,–,–,'首都'
4,0.7,5.6,13.4,0.2,80.1,–,–,–,–,'，'
5,0.7,2.6,41.6,8.8,37.1,9.2,–,–,–,'巴黎'
6,0.4,1.4,35.5,0.2,49.7,1.5,11.3,–,–,'是'
7,2.2,2.5,10.3,5.1,20.6,42.7,12.3,4.4,–,'法国'
8,1.6,10.6,21.2,0.6,22.5,0.5,16.9,0.4,25.8,'的'


### 7.7 加权求和 V，然后 o_proj

$$\text{output} = A V$$

`[B, 16, S, S] @ [B, 16, S, 128] → [B, 16, S, 128]`。第二个 S 被消掉了。

之后要把 16 个头拼回一条向量再过 `o_proj`：

```text
[B, 16, S, 128]
    ↓ transpose(1, 2)      把 S 换回第 1 维
[B, S, 16, 128]
    ↓ contiguous().reshape  16×128 = 2048 拼平
[B, S, 2048]
    ↓ o_proj                2048 → 1024
[B, S, 1024]
```

`transpose` 之后必须 `contiguous()` 才能 `reshape`，因为 transpose 只改了 stride 没搬内存。
这一步的顺序不能反：先 transpose 再 reshape，才能保证同一个位置的 16 个头被拼在一起。

In [150]:
attn_out_heads = torch.matmul(attn_weights, v_rep)

attn_out_merged = attn_out_heads.transpose(1, 2).contiguous().reshape(BATCH, SEQ, -1)

attn_output = my_linear(attn_out_merged, P0.o_proj)

check(attn_output, L0.o_proj, f'L{LAYER} o_proj')
# Attention 模块的最终输出就是 o_proj 的输出，工具已经把 tuple 拆开了
check(attn_output, L0.attn_out, f'L{LAYER} self_attn 输出');


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 o_proj
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 self_attn 输出


### 7.8 第一个残差连接

$$h = x + \mathrm{Attention}(\mathrm{RMSNorm}(x))$$

加的是 §7.1 记下来的 `residual_1`，也就是**未经归一化**的 layer 输入。

In [151]:
hidden_after_attn = residual_1 + attn_output
residual_2 = hidden_after_attn

show([('residual（layer 输入）', residual_1[0, -1].norm().item()),
      ('attention 输出', attn_output[0, -1].norm().item()),
      ('相加之后', hidden_after_attn[0, -1].norm().item())],
     header=['张量', 'L2 范数'], fmt='{:.4f}',
     title='残差前后的量级对比（最后位置）')

look(hidden_after_attn, 'after residual #1')
# 工具在 hook 里也拼了同一个中间态，顺带确认两边一致
check(hidden_after_attn, L0.after_attn, f'L{LAYER} residual add #1');

张量,L2 范数
residual（layer 输入）,0.8380
attention 输出,5.5856
相加之后,5.5742


形状,"(1, 9, 1024)"
"[0, -1, :6]","[-0.6333, -0.3618, -0.0587, -0.1963, +0.1306, +0.1057]"


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 residual add #1


### 7.9 MLP：SwiGLU

$$\mathrm{MLP}(x) = W_{\text{down}} \big( \mathrm{SiLU}(W_{\text{gate}} x) \odot W_{\text{up}} x \big)$$

三个投影，两条并行的路径：

```text
        x  [B, S, 1024]
        ├──── gate_proj ───→ [B, S, 3072] ──→ SiLU ──┐
        │                                            ⊙  逐元素相乘
        └──── up_proj ─────→ [B, S, 3072] ───────────┘
                                    │
                              down_proj
                                    ▼
                             [B, S, 1024]
```

`gate` 和 `up` 是两个**独立的**权重矩阵，不是同一个矩阵切两半。
激活只作用在 gate 分支上，up 分支保持线性。

In [152]:
h2 = my_rmsnorm(hidden_after_attn, P0.norm2)
check(h2, L0.norm2, f'L{LAYER} post_attn_norm')

gate = my_linear(h2, P0.gate)
up = my_linear(h2, P0.up)
check(gate, L0.gate, f'L{LAYER} gate_proj')
check(up, L0.up, f'L{LAYER} up_proj')

activated = my_silu(gate)
gated = activated * up

mlp_output = my_linear(gated, P0.down)
check(mlp_output, L0.down, f'L{LAYER} down_proj')
check(mlp_output, L0.mlp_out, f'L{LAYER} mlp 输出');


✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 post_attn_norm
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 gate_proj
✓  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00  L0 up_proj
✓  max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08  L0 down_proj
✓  max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08  L0 mlp 输出


In [153]:
show([(name, t[0, -1].float().min().item(), t[0, -1].float().max().item(),
       t[0, -1].float().mean().item(), t[0, -1].float().std().item())
      for name, t in [('gate（激活前）', gate), ('SiLU(gate)', activated),
                      ('up', up), ('相乘之后', gated)]],
     header=['张量', 'min', 'max', 'mean', 'std'], fmt='{:+.4f}',
     title='gate 分支与 up 分支的数值分布（最后位置，3072 维）')

_negative_ratio = (gate[0, -1] < 0).float().mean().item()
print(f'gate 为负的比例: {_negative_ratio:.1%}  ← SiLU 把负值压向 0，起到门控作用')

张量,min,max,mean,std
gate（激活前）,-4.1200,+2.4567,-0.5787,+0.7180
SiLU(gate),-0.2785,+2.2627,-0.1244,+0.1570
up,-2.9889,+1.7294,-0.0079,+0.2873
相乘之后,-3.6989,+0.7434,-0.0003,+0.0962


gate 为负的比例: 83.5%  ← SiLU 把负值压向 0，起到门控作用


### 7.10 第二个残差连接，Layer 0 完成

In [154]:
layer_output = residual_2 + mlp_output

check(layer_output, L0.out, f'L{LAYER} layer 输出')
# 下一层的 inp 就是这一层的 out，顺带确认这条链是连着的
check(layer_output, qwen.L[LAYER + 1].inp, f'L{LAYER} 输出 == L{LAYER + 1} 输入');


✓  max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08  L0 layer 输出
✓  max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08  L0 输出 == L1 输入


Layer 0 到此走完。整个 Layer 是保形的：进去 `[1, 9, 1024]`，出来 `[1, 9, 1024]`，
中间的宽度变化归成三段：

- **升维**：1024 → 2048（q_proj）或 1024 → 3072（gate/up_proj）；
- **序列维出现两次**：`[B, 16, S, S]` 是唯一一处形状与序列长度成平方关系的张量；
- **降回 1024**：o_proj 和 down_proj 把宽度还原，残差才能相加。


## 8. 封装成模块

把上面验证过的函数组装成与 Qwen3 源码结构对应的类：

```text
MyQwen3ForCausalLM
└── MyQwen3Model
    ├── my_embedding
    ├── MyQwen3DecoderLayer × 28
    │   ├── MyQwen3RMSNorm      (input_layernorm)
    │   ├── MyQwen3Attention
    │   │   ├── MyQwen3RMSNorm  (q_norm / k_norm)
    │   │   └── RoPE / GQA / causal attention
    │   ├── MyQwen3RMSNorm      (post_attention_layernorm)
    │   └── MyQwen3MLP
    ├── MyQwen3RMSNorm          (final norm)
    └── lm_head（与 embedding 共享权重）
```

参数全部从 §2.2 的目录 `W` 取（`W.L[i].q_proj` 等），拿到的就是真实模型的张量引用，**不复制、不重新初始化**。

每个类的构造参数从"一个源模块"改成"一个层号"，因为 `W.L[i]` 已经把那一层的 11 个权重摆好了。

In [155]:
class MyQwen3RMSNorm:
    """对应 Qwen3RMSNorm。"""

    def __init__(self, weight, eps=RMS_EPS):
        self.weight = weight
        self.eps = eps

    def __call__(self, x):
        return my_rmsnorm(x, self.weight, self.eps)


class MyQwen3MLP:
    """对应 Qwen3MLP，SwiGLU 结构。权重取自 W.L[index]。"""

    def __init__(self, index):
        p = W.L[index]
        self.gate_weight = p.gate
        self.up_weight = p.up
        self.down_weight = p.down

    def __call__(self, x):
        gate = my_linear(x, self.gate_weight)
        up = my_linear(x, self.up_weight)
        return my_linear(my_silu(gate) * up, self.down_weight)

In [156]:
class MyQwen3Attention:
    """对应 Qwen3Attention + eager_attention_forward。"""

    def __init__(self, index):
        p = W.L[index]
        self.q_weight = p.q_proj
        self.k_weight = p.k_proj
        self.v_weight = p.v_proj
        self.o_weight = p.o_proj
        self.q_norm = MyQwen3RMSNorm(p.q_norm)
        self.k_norm = MyQwen3RMSNorm(p.k_norm)

    def __call__(self, x, cos, sin, mask, collect=None):
        batch, seq_len, _ = x.shape
        head_shape = (batch, seq_len, -1, HEAD_DIM)

        # 投影 → 拆头 → 逐头 norm → 头维提前
        q = self.q_norm(my_linear(x, self.q_weight).view(head_shape)).transpose(1, 2)
        k = self.k_norm(my_linear(x, self.k_weight).view(head_shape)).transpose(1, 2)
        v = my_linear(x, self.v_weight).view(head_shape).transpose(1, 2)

        q, k = my_apply_rope(q, k, cos, sin)          # RoPE 在 norm 之后
        k = my_repeat_kv(k, N_REP)                    # GQA 在 RoPE 之后
        v = my_repeat_kv(v, N_REP)

        scores = torch.matmul(q, k.transpose(2, 3)) * SCALING
        if mask is not None:
            scores = scores + mask
        weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)

        out = torch.matmul(weights, v).transpose(1, 2).contiguous()
        out = out.reshape(batch, seq_len, -1)
        if collect is not None:
            collect['attn_weights'] = weights
        return my_linear(out, self.o_weight)

In [157]:
class MyQwen3DecoderLayer:
    """对应 Qwen3DecoderLayer。pre-norm + 两次残差。"""

    def __init__(self, index):
        p = W.L[index]
        self.input_layernorm = MyQwen3RMSNorm(p.norm1)
        self.self_attn = MyQwen3Attention(index)
        self.post_attention_layernorm = MyQwen3RMSNorm(p.norm2)
        self.mlp = MyQwen3MLP(index)

    def __call__(self, hidden_states, cos, sin, mask, collect=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, cos, sin, mask, collect)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        return residual + hidden_states


class MyQwen3Model:
    """对应 Qwen3Model：embedding + 28 层 + final norm。"""

    def __init__(self):
        self.embed_weight = W.embed
        self.layers = [MyQwen3DecoderLayer(i) for i in range(N_LAYERS)]
        self.norm = MyQwen3RMSNorm(W.final_norm)

    def __call__(self, ids, collect_attn=False):
        seq_len = ids.shape[1]
        position_ids = torch.arange(seq_len, device=ids.device).unsqueeze(0)
        cos, sin, _ = my_rope_tables(position_ids)
        mask = my_causal_mask(seq_len)

        hidden = my_embedding(ids, self.embed_weight)
        all_hidden = [hidden]
        all_attn = []
        for layer in self.layers:
            bucket = {} if collect_attn else None
            hidden = layer(hidden, cos, sin, mask, bucket)
            all_hidden.append(hidden)
            if collect_attn:
                all_attn.append(bucket['attn_weights'])
        return self.norm(hidden), all_hidden, all_attn

In [158]:
class MyQwen3ForCausalLM:
    """对应 Qwen3ForCausalLM。lm_head 与 embedding 共享权重。"""

    def __init__(self):
        self.model = MyQwen3Model()
        # 权重绑定：lm_head 与 embedding 是同一个张量，直接引用，不复制
        self.lm_head_weight = W.embed

    def __call__(self, ids, collect_attn=False):
        last_hidden, all_hidden, all_attn = self.model(ids, collect_attn)
        logits = my_linear(last_hidden, self.lm_head_weight)
        return logits, last_hidden, all_hidden, all_attn


my_model = MyQwen3ForCausalLM()
print(f'复现模型层数: {len(my_model.model.layers)}')
print(f'lm_head 权重与 embedding 是同一对象: '
      f'{my_model.lm_head_weight is my_model.model.embed_weight}')
print(f'与官方模型共享权重（未复制）: '
      f'{my_model.model.layers[0].mlp.gate_weight is model.model.layers[0].mlp.gate_proj.weight}')

复现模型层数: 28
lm_head 权重与 embedding 是同一对象: True
与官方模型共享权重（未复制）: True


## 9. 跑完整个模型，与官方逐处比对

零件齐了。这一节按数据流的顺序走一遍：跑出复现的全部中间态，逐层对齐，过出口，
最后落到预测本身。


### 9.1 跑一遍

关键设计：**每层都用自己上一层的输出作为输入**，不从官方结果里"借"中间态。
`cos` / `sin` / mask 也由这一遍自己算。整条链独立向前推进，只在比对时才读官方中间态。
如果每层都拿 `qwen.L[i].inp` 当输入，每层误差都会被重置，验证的强度大打折扣。


In [159]:
my_logits, _, my_all_hidden, my_all_attn = my_model(
    input_ids, collect_attn=True)

print(f'my_logits        {tuple(my_logits.shape)}')
print(f'my_all_hidden    {len(my_all_hidden)} 个 × {tuple(my_all_hidden[0].shape)}')
print(f'my_all_attn      {len(my_all_attn)} 个 × {tuple(my_all_attn[0].shape)}')

my_logits        (1, 9, 151936)
my_all_hidden    29 个 × (1, 9, 1024)
my_all_attn      28 个 × (1, 16, 9, 9)


### 9.2 28 层逐层对齐


In [160]:
# 逐层对齐：hidden state 与 attention 权重，基准一律用 qwen.L[i].out。
layer_errors = []
first_failure = None

for layer_index in range(N_LAYERS):
    mine = my_all_hidden[layer_index + 1]
    ref = qwen.L[layer_index].out
    diff = (mine.float() - ref.float()).abs()
    max_abs = diff.max().item()
    mean_abs = diff.mean().item()
    scale_rel = max_abs / ref.float().abs().max().item()

    attn_diff = (my_all_attn[layer_index].float()
                 - qwen.L[layer_index].attn_weights.float()).abs().max().item()

    ok = scale_rel < REL_THRESHOLD and attn_diff < REL_THRESHOLD
    if not ok and first_failure is None:
        first_failure = layer_index
    layer_errors.append((layer_index, max_abs, mean_abs, scale_rel, attn_diff, ok))

_all_ok = all(row[5] for row in layer_errors)
show(layer_errors,
     header=['层', 'hidden max_abs', 'hidden mean_abs', 'scale_rel',
             'attn max_abs', 'ok'],
     title=f'全部 {N_LAYERS} 层通过（scale_rel < {REL_THRESHOLD:.0e}）: {_all_ok}'
           + (f'\n⚠ 首个不一致的层: Layer {first_failure}'
              if first_failure is not None else ''))

# 把上表实测到的最大值交给汇总，别让汇总里出现假的 0.000e+00。
# attention 权重的 max|ref| 恒为 1（位置 0 只能看自己），所以它的绝对差就是相对差。
_attn_worst = max(row[4] for row in layer_errors)
record(f'全部 {N_LAYERS} 层 hidden state', _all_ok,
       max(row[1] for row in layer_errors), max(row[3] for row in layer_errors))
record(f'全部 {N_LAYERS} 层 attention 权重',
       _attn_worst < REL_THRESHOLD, _attn_worst, _attn_worst);

层,hidden max_abs,hidden mean_abs,scale_rel,attn max_abs,ok
0,2.384e-07,1.465e-08,3.891e-08,0.000e+00,✓
1,1.192e-06,6.816e-08,1.344e-07,8.643e-07,✓
2,1.465e-03,3.720e-07,2.291e-07,1.252e-06,✓
3,1.465e-03,4.077e-07,2.292e-07,7.153e-07,✓
4,1.465e-03,4.383e-07,2.292e-07,8.941e-07,✓
5,1.465e-03,4.893e-07,2.292e-07,7.153e-07,✓
6,1.465e-03,5.390e-07,2.293e-07,1.341e-06,✓
7,1.465e-03,5.955e-07,2.293e-07,9.537e-07,✓
8,1.465e-03,6.520e-07,2.295e-07,1.192e-06,✓
9,1.465e-03,7.016e-07,2.297e-07,1.192e-06,✓


### 9.3 出口：final norm → LM Head

```text
   Layer 27 输出  [B, S, 1024]
         ↓
   final RMSNorm                    model.model.norm
         ↓        [B, S, 1024]
         ↓
   lm_head        ← 复用 embedding 那张 [151936, 1024] 表
         ↓
      logits      [B, S, 151936]
```

LM Head 的计算就是 `hidden @ embed_weight.T`：把 1024 维的 hidden state 与词表里 151936 个
token 向量逐个做内积。**内积大 = 方向接近 = 得分高。**
入口那次查表和出口这次打分，用的是同一张矩阵。


In [161]:
my_final_hidden = my_model.model.norm(my_all_hidden[-1])

check(my_final_hidden, qwen.final_norm, 'final RMSNorm')
check(my_logits, qwen.logits, 'lm_head logits')

print()
print(f'官方 logits 数值范围: [{qwen.logits.min().item():.3f}, '
      f'{qwen.logits.max().item():.3f}]')
print(f'复现 logits 数值范围: [{my_logits.min().item():.3f}, {my_logits.max().item():.3f}]')

✓  max_abs=8.297e-05  mean_abs=2.408e-06  max_rel=1.051e-06  final RMSNorm
✓  max_abs=3.242e-05  mean_abs=3.097e-06  max_rel=1.373e-06  lm_head logits

官方 logits 数值范围: [-18.614, 23.609]
复现 logits 数值范围: [-18.614, 23.609]


### 9.4 预测是否一致

逐节点比对之外，还有一个更直接的确认：**预测本身是否一致**。
先看最后一个位置的 Top-10，再看全部 9 个位置的 argmax。


In [162]:
TOP_K = 10
last_position = SEQ - 1

official_last = qwen.logits[0, last_position]
my_last = my_logits[0, last_position]

official_top = official_last.topk(TOP_K)
my_top = my_last.topk(TOP_K)

official_probs = torch.softmax(official_last, dim=-1)

_rows = []
for rank in range(TOP_K):
    o_id = official_top.indices[rank].item()
    m_id = my_top.indices[rank].item()
    _rows.append((rank + 1,
                  repr(tokenizer.decode([o_id])), official_top.values[rank].item(),
                  f'{official_probs[o_id].item():.2%}',
                  repr(tokenizer.decode([m_id])), my_top.values[rank].item(),
                  o_id == m_id))

show(_rows, header=['排名', '官方 token', 'logit', '概率',
                    '复现 token', 'logit', '一致'],
     fmt='{:.3f}',
     title=f'输入: {PROMPT}\n'
           f'预测第 {SEQ + 1} 个 token（基于位置 {last_position} 的 logits）')

排名,官方 token,logit,概率,复现 token,logit,一致
1,'首都',23.249,95.94%,'首都',23.249,✓
2,'首',19.600,2.50%,'首',19.600,✓
3,'象征',17.285,0.25%,'象征',17.285,✓
4,'都',17.098,0.20%,'都',17.098,✓
5,'国家',16.898,0.17%,'国家',16.898,✓
6,'____',15.765,0.05%,'____',15.765,✓
7,'代表',15.759,0.05%,'代表',15.759,✓
8,'代',15.716,0.05%,'代',15.716,✓
9,'省',15.660,0.05%,'省',15.660,✓
10,'第二',15.353,0.04%,'第二',15.353,✓


每个位置都在预测"它的下一个 token"，这是 causal 语言模型的定义。位置 8（最后一个 `的`）
的预测才是我们关心的续写结果，前面几个位置顺带展示了模型读到一半时的判断。


In [163]:
# 闭环验证：官方与复现的 argmax 预测是否一致（全部 9 个位置）
official_argmax = qwen.logits[0].argmax(-1)
my_argmax = my_logits[0].argmax(-1)
all_match = torch.equal(official_argmax, my_argmax)

show([(position,
       repr(tokenizer.decode([input_ids[0, position].item()])),
       repr(tokenizer.decode([official_argmax[position].item()])),
       repr(tokenizer.decode([my_argmax[position].item()])),
       official_argmax[position].item() == my_argmax[position].item())
      for position in range(SEQ)],
     header=['位置', '输入 token', '官方预测', '复现预测', '一致'],
     title=f'全部 {SEQ} 个位置预测一致: {all_match}')

record(f'全部 {SEQ} 个位置 argmax 一致', all_match);

位置,输入 token,官方预测,复现预测,一致
0,'北京','地铁','地铁',✓
1,'是中国','的','的',✓
2,'的','首都','首都',✓
3,'首都','，','，',✓
4,'，','也是','也是',✓
5,'巴黎','是','是',✓
6,'是','法国','法国',✓
7,'法国','的','的',✓
8,'的','首都','首都',✓


In [164]:
# 把预测接回原文，完成一次完整闭环
next_token_id = official_argmax[last_position].item()
continuation = tokenizer.decode([next_token_id])

print(f'原文:   {PROMPT}')
print(f'续写:   {PROMPT}{continuation}')
print()
print(f'预测 token id = {next_token_id}, 文本 = {continuation!r}, '
      f'概率 = {official_probs[next_token_id].item():.2%}')


原文:   北京是中国的首都，巴黎是法国的
续写:   北京是中国的首都，巴黎是法国的首都

预测 token id = 106114, 文本 = '首都', 概率 = 95.94%


### 9.5 验证汇总

`summary()` 报的是**本次记录到什么**，不是"该验证的都验证了"——后者它无从知晓：
漏跑一格，分子分母一起少一项，`通过 N / N` 照旧成立。
完整性由另一件事证明：从头 Run All 之后 `execution_count` 连续无缺口。


In [165]:
summary();

max_abs,max_rel,ok,检查项
0.000e+00,0.000e+00,✓,官方 L0 输出直连 L1 输入
0.000e+00,0.000e+00,✓,my_linear vs q_proj
0.000e+00,0.000e+00,✓,my_rmsnorm vs input_layernorm
0.000e+00,0.000e+00,✓,my_rmsnorm vs post_attn_norm
1.192e-07,6.356e-08,✓,my_silu vs F.silu
0.000e+00,0.000e+00,✓,my_embedding vs embed_tokens
0.000e+00,0.000e+00,✓,my_embedding vs Layer 0 输入
0.000e+00,0.000e+00,✓,my_position_ids
0.000e+00,0.000e+00,✓,my_rope cos
0.000e+00,0.000e+00,✓,my_rope sin


In [166]:
# 端到端的最终确认
_end_to_end = [
    ('logits 形状一致', tuple(my_logits.shape) == tuple(qwen.logits.shape)),
    ('argmax 预测全部一致', torch.equal(my_logits[0].argmax(-1),
                                    qwen.logits[0].argmax(-1))),
    (f'Top-{TOP_K} 排序一致', torch.equal(my_last.topk(TOP_K).indices,
                                      official_last.topk(TOP_K).indices)),
    (f'logits 相对误差 < {REL_THRESHOLD:.0e}',
     (my_logits - qwen.logits).abs().max().item()
     / qwen.logits.abs().max().item() < REL_THRESHOLD),
    ('28 层 hidden state 全部对齐', all(row[5] for row in layer_errors)),
    ('28 层 attention 权重全部对齐',
     all(row[4] < REL_THRESHOLD for row in layer_errors)),
]
for name, ok in _end_to_end:
    print(f'{"✓" if ok else "✗"} {name}')
print()
print(f'端到端闭环成立: {all(ok for _, ok in _end_to_end)}')

✓ logits 形状一致
✓ argmax 预测全部一致
✓ Top-10 排序一致
✓ logits 相对误差 < 1e-05
✓ 28 层 hidden state 全部对齐
✓ 28 层 attention 权重全部对齐

端到端闭环成立: True


## 10. 实验发现 / Experiment Findings

以下内容由实际运行数据产生，不是预先写好的结论。

### 10.1 复现与官方一致

判据固定为 `max|差| / max|ref| < 1e-5`，全程没有可调公差。本次 Run All 记录到 40 项验证，全部通过。

- **Layer 0 内部 22 个节点**逐元素比对通过。其中 `q_proj`、`k_proj`、`v_proj`、`q_norm`、`k_norm`、
  RoPE、`repeat_kv`、attention 权重、`o_proj`、`gate_proj`、`up_proj` 与官方**逐位相同**（`max_abs = 0`）。
- **28 层的 hidden state 与 attention 权重全部对齐。** 第二遍是独立向前推进的：
  每层吃自己上一层的输出，`cos` / `sin` / mask 也由第二遍自己算，只在比对时才读官方中间态。
- **9 个位置的 argmax 预测与 Top-10 排序完全一致**，官方与复现的 logits 数值范围同为 `[-18.614, 23.609]`。

`QK^T`、`+ mask`、`A @ V` 这三个中间量官方不单独暴露，没有对照物，只能靠下游节点间接验证。


### 10.2 三个会让人写错代码的坑

这三条不是"知识点"，是本次真的踩到、并且不看报错读不出来的东西。

**（1）`rope_theta` 不在 `config` 顶层。**
`config.json` 里它是顶层字段，但 transformers 5.15 把它收进了 `config.rope_parameters` 字典：
`hasattr(config, 'rope_theta')` 返回 **False**，实测值要从 `config.rope_parameters['rope_theta']` 取。
凭记忆写 `config.rope_theta` 直接 `AttributeError`。

**（2）`q_norm` / `k_norm` 作用的维度比预想的靠里。**
它归一化的是 **reshape 之后**的 `head_dim=128` 维，不是 1024 维；`weight` 长度只有 128，
被 16 个 q 头（或 8 个 kv 头）共享；V 完全没有 `norm`（`hasattr(self_attn, 'v_norm')` 为 False）。
源码把 projection、view、norm、transpose 四个操作压在一行里，很容易读成"在 1024 维上归一化"。

**（3）RMSNorm 的两个参数传反了不报错。**
`weight` 是 `(1024,)`、输入是 `(1, 9, 1024)`，两者广播成功，
输出形状与正确写法**一模一样**，但归一化的对象变成了权重。
没有异常、没有形状不符，只有数值是错的——只能靠 `check` 发现。
所以 `my_rmsnorm` 在入口断言 `weight.dim() == 1`。


### 10.3 两条与直觉相反的结构事实

- **出口那张打分矩阵在磁盘上没有自己的副本。** `model.safetensors` 的 310 个键里没有
  `lm_head.weight`，只有 `model.embed_tokens.weight`；`lm_head` 的权重是加载时按
  `_tied_weights_keys`（`{'lm_head.weight': 'model.embed_tokens.weight'}`）绑上去的，
  `data_ptr()` 与 embedding 相同。这张 `[151936, 1024]` = 155,582,464 个参数
  占总量 596,049,920 的 **26.1%**，共享张量两边都只数一次，
  所以文件内参数量与 `model.parameters()` 的计数正好相等。
- **`head_dim` 与 `hidden_size / num_heads` 无关。** 这里 `1024 / 16 = 64`，
  而 `config.head_dim` 写死是 **128**，所以 `q_proj` 把 1024 **升到了 2048**，比残差流还宽。
  按"head_dim = hidden / heads"推，每一个形状都会算错。


## 11. 小结

### 11.1 §0 的问题，现在可以回答

> Qwen3-0.6B 对一段真实 token 序列究竟进行了哪些计算，这些计算能否被我们自己重新实现并验证？

能。实验 1 给出了顶层骨架，§7 逐节点拆开了 Layer 0，§9 把 28 层跑完——
每一步都用 PyTorch 基础算子重写、与官方实现比对通过。

这条链用到的算子就这么几个：`matmul`、逐元素乘加、`rsqrt`、`sigmoid`、`softmax`、`cat`，
加上 `view` / `transpose` / `reshape` / `expand` 这些只改视图、不动数值的形变。
一个 6 亿参数的语言模型，计算上就是这些东西按固定顺序叠 28 遍。

### 11.2 这次验证证明了什么，没证明什么

**证明了**：在本次这一个输入上，第二遍的 28 层实现与官方逐节点一致，
9 个位置的 argmax 与 Top-10 排序完全一致。第二遍是**独立向前推进**的——
每层吃自己上一层的输出，只在比对时才读官方中间态，所以这不是"抄一步对一步"。

**没证明**：验证只覆盖了一个点，不是一个范围。

| 只测了 | 没测 |
|---|---|
| `PROMPT` 这一个 9 token 输入 | 其他句子、其他长度 |
| `BATCH = 1` | 批量 > 1 时的广播与 mask |
| float32、CPU | bfloat16、GPU |
| `use_cache=False` 的 prefill | KV Cache 的增量 decode 路径（`position_ids` 不再从 0 开始） |

### 11.3 留给下一个实验

- **KV Cache。** 本实验每次都重算整段序列。加上 cache 之后，
  `position_ids` 与 mask 的形状都会变，是第一个会打破上面那张表的东西。
- **采样。** 本实验只取 argmax。`'首都'` 拿到 95.94%、与第二名差 3.649 个 logit，
  这种情况下 temperature / top-p 几乎不起作用；换一个分布平坦的位置才看得出区别。
- **换一个输入重跑。** 上面那张表里"没测"的每一格，都可以用一次重跑变成"测了"。
  最省事的是先换句子和长度——`SEQ` 一变，mask 与 `position_ids` 的形状跟着变，
  是最容易暴露硬编码的地方。
